<a href="https://colab.research.google.com/github/DeveloperDevc/student-performance-analytics/blob/main/Pythonwithaiproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import numpy as np

def load_student_data(file_path):
    try:
        df = pd.read_csv(file_path)
        print(f"Successfully loaded spreadsheet: '{file_path}' ({len(df)} records found)")
        return df
    except FileNotFoundError:
        print(f"Error: The spreadsheet '{file_path}' was not found.")
        return None

def calculate_student_metrics(df, components):
    recalculated_scores = []

    for _, row in df.iterrows():
        comp_marks = [row[comp] for comp in components]
        avg_score = sum(comp_marks) / len(comp_marks)
        recalculated_scores.append(round(avg_score, 2))

    df['Calculated_Average'] = recalculated_scores
    return df

def assign_grade(score):
    if score >= 85:
        return 'A'
    elif score >= 70:
        return 'B'
    elif score >= 50:
        return 'C'
    elif score >= 40:
        return 'D'
    else:
        return 'F'

def determine_pass_fail(row, min_score=40, min_attendance=50):
    final_score = row['FinalScore_100']
    attendance = row['Attendance_%']

    if final_score >= min_score and attendance >= min_attendance:
        return 'Pass'
    else:
        return 'Fail'

def process_grades_and_status(df):
    grades = []
    statuses = []

    for _, row in df.iterrows():
        score = row['FinalScore_100']
        grades.append(assign_grade(score))
        statuses.append(determine_pass_fail(row))

    df['Computed_Grade'] = grades
    df['Computed_Status'] = statuses
    return df

def generate_component_analysis(df, components):
    analysis_data = {}

    for comp in components:
        analysis_data[comp] = {
            'Highest': df[comp].max(),
            'Lowest': df[comp].min(),
            'Average': round(df[comp].mean(), 2),
            'Std Dev': round(df[comp].std(), 2)
        }

    return pd.DataFrame(analysis_data).T

def get_top_performers(df, n=5):
    return df.nlargest(n, 'FinalScore_100')[['StudentID', 'Name', 'Attendance_%', 'FinalScore_100', 'Grade', 'Status']]

def main():
    input_csv = 'student_performance_data.csv'
    output_csv = 'student_performance_analyzed_report.csv'

    components = ['Attendance_%', 'Assignments_100', 'Midterm_100', 'FinalExam_100', 'FinalScore_100']

    print("-" * 70)
    print("      STUDENT PERFORMANCE ANALYTICS SYSTEM (CSV SPREADSHEET)")
    print("-" * 70)

    df = load_student_data(input_csv)
    if df is None:
        return

    df = calculate_student_metrics(df, ['Assignments_100', 'Midterm_100', 'FinalExam_100'])
    df = process_grades_and_status(df)

    print("\n 1. COMPONENT-WISE PERFORMANCE ANALYSIS ")
    component_summary = generate_component_analysis(df, components)
    print(component_summary.to_string(), "\n")

    print(" 2. CLASS OVERALL PERFORMANCE SUMMARY ")
    total_students = len(df)
    passed_students = (df['Computed_Status'] == 'Pass').sum()
    failed_students = total_students - passed_students
    pass_percentage = (passed_students / total_students) * 100

    print(f"Total Students Evaluated : {total_students}")
    print(f"Passed Students          : {passed_students} ({pass_percentage:.2f}%)")
    print(f"Failed Students          : {failed_students} ({100 - pass_percentage:.2f}%)")
    print(f"Class Average Final Score: {df['FinalScore_100'].mean():.2f}")
    print(f"Class Average Attendance : {df['Attendance_%'].mean():.2f}%\n")

    print(" 3. GRADE DISTRIBUTION ")
    print(df['Computed_Grade'].value_counts().sort_index().to_string(), "\n")

    print(" 4. TOP 5 PERFORMERS ")
    top_students = get_top_performers(df, n=5)
    print(top_students.to_string(index=False), "\n")

    df.to_csv(output_csv, index=False)
    print(f"Processed analytics successfully exported to CSV spreadsheet: '{output_csv}'.")
    print("-" * 70)

if __name__ == "__main__":
    main()

----------------------------------------------------------------------
      STUDENT PERFORMANCE ANALYTICS SYSTEM (CSV SPREADSHEET)
----------------------------------------------------------------------
Successfully loaded spreadsheet: 'student_performance_data.csv' (100 records found)

 1. COMPONENT-WISE PERFORMANCE ANALYSIS 
                 Highest  Lowest  Average  Std Dev
Attendance_%       100.0    60.0    81.39    11.89
Assignments_100    100.0    50.0    77.17    14.60
Midterm_100        100.0    45.0    74.01    16.53
FinalExam_100      100.0    36.0    75.90    17.29
FinalScore_100      97.5    51.9    76.14    11.94 

 2. CLASS OVERALL PERFORMANCE SUMMARY 
Total Students Evaluated : 100
Passed Students          : 100 (100.00%)
Failed Students          : 0 (0.00%)
Class Average Final Score: 76.14
Class Average Attendance : 81.39%

 3. GRADE DISTRIBUTION 
Computed_Grade
A    28
B    41
C    31 

 4. TOP 5 PERFORMERS 
StudentID          Name  Attendance_%  FinalScore_100 Grade 